# Bronze Layer — Payment Events

**Table**: `aiubereats.payments.bronze_payment_events`  
**Source**: `/Volumes/aiubereats/payments/raw_json/` (100 JSON files)  
**Purpose**: Raw ingestion, idempotent MERGE on event_id, schema-on-read, full source fidelity.  
**Quality level**: As-is from source — no transformation beyond ingestion metadata.

## Design Decisions

| Decision | Choice | Rationale |
|----------|--------|-----------|
| Schema | Explicit `StructType` | The source schema is known and stable. Explicit schema prevents `event.timestamp` being inferred as `DoubleType` due to scientific notation (e.g. `1.7596876023E12`), which would silently lose precision. Schema-on-read inference would mis-type this field. |
| Scientific notation | Read as `StringType`, cast to `LongType` in Silver | Bronze preserves the raw value. The cast is a transformation and belongs in Silver. |
| `event.timestamp` type in Bronze | `LongType` via explicit schema with `allowNumericLeadingZeros` + `PERMISSIVE` mode | Spark's JSON reader, when given an explicit `LongType`, will parse `1.7596876023E12` correctly via its numeric coercion. This is validated below. |
| Write mode | `MERGE on event_id` | Idempotent: new event_ids are inserted, existing event_ids are skipped. Re-runs are safe. |
| Partitioning | `_ingested_date` (DATE derived from `_ingested_at`) | Enables cheap partition pruning when Silver reads only recent Bronze data. Day-level granularity is appropriate for a batch-loaded payment event dataset. |
| `mergeSchema` | `true` | Defensive setting — allows the source to add new fields without breaking ingestion. New fields land in Bronze and are evaluated at the Silver transition. |
| Serverless / Spark Connect | No RDD, no `sparkContext`, no `mapPartitions` | All operations use the DataFrame API only. `spark.read.format('json')` is preferred over `spark.read.json()` for explicitness and option-setting consistency. |


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, to_date
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, TimestampType
)
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

In [ ]:
# ── Configuration — Databricks Widgets ───────────────────────────────────────────────
#
# Parameters are declared as widgets so they can be overridden at runtime
# (via Databricks UI, Jobs API, or dbutils.widgets.set() in a calling notebook).
# dbutils.widgets.text(name, defaultValue) is idempotent — safe to re-run.

dbutils.widgets.text("bronze_table",  "aiubereats.payments.bronze_payment_events")
dbutils.widgets.text("source_path",   "/Volumes/aiubereats/payments/raw_json/")
dbutils.widgets.text("source_system", "uber_eats_payments_api")

BRONZE_TABLE  = dbutils.widgets.get("bronze_table")
SOURCE_PATH   = dbutils.widgets.get("source_path")
SOURCE_SYSTEM = dbutils.widgets.get("source_system")

In [ ]:
# ── Ensure catalog and schema exist ──────────────────────────────────────────
#
# Databricks Serverless jobs run in a clean context with no default catalog or
# schema. These statements are idempotent — safe to re-run on every execution.

spark.sql("CREATE CATALOG IF NOT EXISTS aiubereats")
spark.sql("CREATE SCHEMA IF NOT EXISTS aiubereats.payments")

In [ ]:
# ── Create Bronze table (DDL) ─────────────────────────────────────────────────
#
# Run once. The PySpark write below uses saveAsTable which would also create it,
# but explicit DDL documents the intended schema and properties up front.
#
# Partitioned by _ingested_date (DATE) for efficient incremental reads from Silver.
# TBLPROPERTIES:
#   autoOptimize.optimizeWrite  — coalesce small files during write (Databricks)
#   autoOptimize.autoCompact    — background compaction (Databricks)
#   enableDeletionVectors       — fast logical deletes without full file rewrite
#   logRetentionDuration        — keep Delta log for 90 days (audit trail)
#   deletedFileRetentionDuration — 7 days before VACUUM can clean old files

spark.sql("""
    CREATE TABLE IF NOT EXISTS aiubereats.payments.bronze_payment_events (
        event_id              STRING,
        payment_id            STRING,
        event                 STRUCT<
            event_name: STRING,
            timestamp:  BIGINT
        >,
        dt_current_timestamp  STRING,
        _ingested_at          TIMESTAMP,
        _ingested_date        DATE,
        _source_file          STRING,
        _source_system        STRING
    )
    USING DELTA
    PARTITIONED BY (_ingested_date)
    TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite'       = 'true',
        'delta.autoOptimize.autoCompact'         = 'true',
        'delta.enableDeletionVectors'            = 'true',
        'delta.logRetentionDuration'             = 'interval 90 days',
        'delta.deletedFileRetentionDuration'     = 'interval 7 days',
        'delta.enableChangeDataFeed'             = 'true',
        'pipelines.channel'                      = 'CURRENT'
    )
    COMMENT 'Bronze layer: raw payment event records ingested as-is from JSON source files.'
""")

In [ ]:
# ── Explicit source schema ────────────────────────────────────────────────────
#
# WHY EXPLICIT SCHEMA:
#   Spark's JSON schema inference reads a sample of records. When it encounters
#   a value like 1.7596876023E12 for event.timestamp, it infers DoubleType.
#   DoubleType has ~15 significant digits of precision; a millisecond-epoch
#   timestamp has 13 digits. In practice, doubles can represent 13-digit integers
#   exactly, but this is fragile and produces a float column rather than an
#   integer column. Providing LongType explicitly:
#     1. Forces numeric coercion during read — Spark's JSON reader handles
#        scientific notation strings ("1.7596876023E12") when the target type
#        is LongType by parsing the float representation and converting to long.
#     2. Prevents precision loss from double arithmetic downstream.
#     3. Documents intent: this field IS a long integer epoch millisecond.

SOURCE_SCHEMA = StructType([
    StructField("event_id",           StringType(), nullable=True),
    StructField("payment_id",         StringType(), nullable=True),
    StructField("event", StructType([
        StructField("event_name",  StringType(), nullable=True),
        StructField("timestamp",   LongType(),   nullable=True),
    ]), nullable=True),
    StructField("dt_current_timestamp", StringType(), nullable=True),
])

In [ ]:
# ── Ingest JSON files into Bronze ─────────────────────────────────────────────
#
# spark.read.format("json") vs spark.read.json():
#   Both call the same underlying reader. format("json") is preferred because:
#     - It is the canonical Spark DataFrameReader pattern, consistent with all
#       other formats (parquet, delta, csv).
#     - .option() chaining is explicit and readable.
#     - spark.read.json() is a convenience shorthand that does not accept the
#       full set of options as cleanly in Spark Connect environments.
#
# mode=PERMISSIVE: malformed records land with nulls rather than failing the job.
#   A _corrupt_record column captures the raw bad line for auditability.
#   Bronze never rejects data — it records what arrived.
#
# Serverless note: input_file_name() is a standard Column function that works
#   with Spark Connect. It resolves to the physical file path per row.

raw_df = (
    spark.read
    .format("json")
    .schema(SOURCE_SCHEMA)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .option("mergeSchema", "true")
    .load(SOURCE_PATH)
)

In [ ]:
# ── Add ingestion metadata columns ────────────────────────────────────────────
#
# _ingested_at   — wall-clock timestamp when this batch ran (TIMESTAMP)
# _ingested_date — DATE derived from _ingested_at; used as partition column
# _source_file   — full path of the source JSON file each row came from
# _source_system — logical system identifier for lineage tracking
#
# _metadata.file_path is the Unity Catalog / Spark Connect equivalent of
# input_file_name(). input_file_name() relies on TaskContext which is not
# available in Serverless (Spark Connect). _metadata is a hidden struct
# automatically injected by the JSON reader and is UC-compatible.

from pyspark.sql.functions import lit

bronze_df = (
    raw_df
    .withColumn("_ingested_at",    current_timestamp())
    .withColumn("_ingested_date",  to_date(current_timestamp()))
    .withColumn("_source_file",    col("_metadata.file_path"))
    .withColumn("_source_system",  lit(SOURCE_SYSTEM))
)

In [ ]:
# ── Structural validation (shift-left, Bronze-level) ──────────────────────────
#
# Bronze does not transform or filter data. It does validate that the source
# is parseable and non-empty before writing. This is a structural check, not
# a business rule check.

row_count = bronze_df.count()
if row_count == 0:
    raise ValueError(
        f"Bronze ingestion aborted: no records found at {SOURCE_PATH}. "
        "Verify the Volume path and that JSON files are present."
    )

print(f"Records to ingest: {row_count:,}")

# Warn if expected columns are absent (structural, not null check)
expected_cols = {"event_id", "payment_id", "event", "dt_current_timestamp"}
actual_cols   = set(raw_df.columns)
missing_cols  = expected_cols - actual_cols
if missing_cols:
    raise ValueError(
        f"Bronze ingestion aborted: source files are missing required columns: {missing_cols}"
    )

print("Structural validation passed.")

In [ ]:
# ── Write to Bronze — MERGE on event_id (idempotent) ────────────────────────
#
# WHY MERGE INSTEAD OF APPEND:
#   append is not idempotent. If the same JSON content arrives under a different
#   filename (re-processed file, copy, retry), append would create duplicate rows
#   for the same event_id. MERGE on event_id guarantees exactly-once semantics:
#     - New event_ids → INSERT
#     - Existing event_ids → SKIP (no update needed in Bronze; raw truth is
#       preserved as-is from the first ingestion of that event_id)
#
# MERGE condition: t.event_id = s.event_id
#   whenNotMatchedInsertAll() — inserts only rows whose event_id is not yet in
#   the table. whenMatched is intentionally absent: Bronze never overwrites a
#   previously ingested raw record (corrections are handled in Silver).
#
# First-run fallback: if the table does not exist yet (DDL cell was skipped),
#   we fall back to saveAsTable with partitionBy so the table is created
#   correctly on the first execution.
#
# Serverless / Spark Connect: DeltaTable.forName() and .merge() are supported
#   in Databricks Serverless (Delta is bundled in Databricks Runtime). No RDD
#   operations are involved — the merge plan is executed as a Spark query.

if spark.catalog.tableExists(BRONZE_TABLE):
    bronze_delta = DeltaTable.forName(spark, BRONZE_TABLE)

    merge_result = (
        bronze_delta.alias("t")
        .merge(
            bronze_df.alias("s"),
            "t.event_id = s.event_id"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"MERGE complete into {BRONZE_TABLE}.")
    print(f"Source rows evaluated: {row_count:,}")
else:
    # First run without prior DDL execution
    (
        bronze_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .partitionBy("_ingested_date")
        .saveAsTable(BRONZE_TABLE)
    )
    print(f"Initial load complete into {BRONZE_TABLE}. {row_count:,} records inserted.")

In [ ]:
# ── Verify ────────────────────────────────────────────────────────────────────
spark.table(BRONZE_TABLE).display()

In [ ]:
# ── Scientific notation sanity check ─────────────────────────────────────────
#
# Confirm that event.timestamp landed as BIGINT (LongType), not DOUBLE.
# If the schema was inferred rather than explicit, this would be DoubleType
# and the value would display with decimal noise.

from pyspark.sql.functions import col

(
    spark.table(BRONZE_TABLE)
    .select(
        col("event_id"),
        col("event.timestamp").alias("raw_event_timestamp"),
        col("_source_file")
    )
    .limit(5)
    .display()
)

# Confirm the column dtype is bigint
ts_field = [
    f for f in spark.table(BRONZE_TABLE).schema["event"].dataType.fields
    if f.name == "timestamp"
][0]
print(f"event.timestamp dtype: {ts_field.dataType}")
assert str(ts_field.dataType) == "LongType()", (
    f"Expected LongType, got {ts_field.dataType}. "
    "Scientific notation values may have been mis-typed."
)